Universal Meteorological Imputation Model
Trains a Bidirectional LSTM on multiple AWS node datasets with ERA5 auxiliary features,
cyclical time encodings, and a Node Embedding Layer.
Evaluates model on simulated missingness (10%-50%) and exports imputed datasets.


In [1]:
import os
import sys
import glob
import warnings
import asyncio

# Fix Windows Proactor event loop issue with ZMQ/TF
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

warnings.filterwarnings('ignore')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


# Environment & Directories Configuration


In [2]:
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')

if IS_KAGGLE:
    print("[INFO] Running in Kaggle Environment")
    # User-specified paths for Kaggle
    user_cache = "/kaggle/input/notebooks/jerismeteo/cek-data-sensor"
    user_era5 = "/kaggle/input/notebooks/jerismeteo/generate-era5-data/cuaca_gabungan_jerukagung.csv"
    
    if os.path.exists(user_cache):
        CACHE_DIR = user_cache
    else:
        input_dirs = glob.glob('/kaggle/input/*/Analisis_Meteorologi')
        if input_dirs:
            BASE_DIR = input_dirs[0]
        else:
            root_inputs = glob.glob('/kaggle/input/*')
            BASE_DIR = root_inputs[0] if root_inputs else '/kaggle/input'
        CACHE_DIR = os.path.join(BASE_DIR, 'cache_data')
        
    if os.path.exists(user_era5):
        PATH_ERA5 = user_era5
    else:
        PATH_ERA5_matches = glob.glob('/kaggle/input/**/cuaca_gabungan_jerukagung.csv', recursive=True)
        if PATH_ERA5_matches:
            PATH_ERA5 = PATH_ERA5_matches[0]
        else:
            PATH_ERA5 = os.path.join(BASE_DIR, 'forecast_open_meteo_jerukangung', 'cuaca_gabungan_jerukagung.csv')
        
    OUTPUT_BASE_DIR = '/kaggle/working'
    OUTPUTS_DIR = os.path.join(OUTPUT_BASE_DIR, 'outputs')
    WRITE_CACHE_DIR = os.path.join(OUTPUT_BASE_DIR, 'cache_data')
else:
    print("[INFO] Running in Local Environment")
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    BASE_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, '..')) if os.path.basename(SCRIPT_DIR) == 'model_imputasi' else SCRIPT_DIR
    CACHE_DIR = os.path.join(BASE_DIR, 'cache_data')
    PATH_ERA5 = os.path.join(BASE_DIR, 'forecast_open_meteo_jerukangung', 'cuaca_gabungan_jerukagung.csv')
    OUTPUTS_DIR = os.path.join(BASE_DIR, 'model_imputasi', 'outputs')
    WRITE_CACHE_DIR = CACHE_DIR

os.makedirs(os.path.join(OUTPUTS_DIR, 'model'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_DIR, 'metrics'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_DIR, 'plots'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_DIR, 'reports'), exist_ok=True)
os.makedirs(WRITE_CACHE_DIR, exist_ok=True)

TARGET_VARS = ['temperature', 'humidity', 'pressure', 'dewpoint']

[INFO] Running in Kaggle Environment


# TensorFlow Check


In [3]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks as tf_callbacks
    print(f'[OK] TensorFlow v{tf.__version__} is active.')
except ImportError:
    print('[ERROR] TensorFlow/Keras is required for this Bidirectional LSTM model.')
    sys.exit(1)


2026-06-20 18:08:46.961899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781978927.171901      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781978927.232289      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781978927.688677      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781978927.688716      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781978927.688719      23 computation_placer.cc:177] computation placer alr

[OK] TensorFlow v2.19.0 is active.


# 1. Ingest & Align AWS Node Data


In [4]:
print('\n[1/10] Scanning and loading AWS node datasets...')
node_files = glob.glob(os.path.join(CACHE_DIR, 'id-*_raw.csv'))
if not node_files:
    raise FileNotFoundError(f"No node CSV files matching 'id-*_raw.csv' found in {CACHE_DIR}")

node_data = {}
global_min_time = None
global_max_time = None

col_map = {
    'dew': 'dewpoint',
    'dewpoint': 'dewpoint',
    'dew_point': 'dewpoint',
    'temp': 'temperature',
    'temperature': 'temperature',
    'humi': 'humidity',
    'humidity': 'humidity',
    'pres': 'pressure',
    'pressure': 'pressure'
}

for fpath in sorted(node_files):
    node_id = os.path.basename(fpath).split('_')[0]
    print(f"  Processing {node_id} from {os.path.basename(fpath)}...")
    
    df_raw = pd.read_csv(fpath)
    if 'timestamp' not in df_raw.columns:
        print(f"    [WARN] No timestamp column found in {node_id}. Skipping.")
        continue
        
    # Convert timestamp to local datetime
    df_raw['datetime'] = (pd.to_datetime(df_raw['timestamp'], unit='s', utc=True)
                          .dt.tz_convert('Asia/Jakarta')
                          .dt.tz_localize(None))
    
    df_raw = (df_raw.dropna(subset=['datetime'])
              .sort_values('datetime')
              .drop_duplicates(subset='datetime')
              .set_index('datetime'))
    
    # Rename key columns for standardization
    df_rename = df_raw.rename(columns=col_map)
    df_rename = df_rename.loc[:, ~df_rename.columns.duplicated()]
    
    # Verify that all 4 target variables exist
    missing_cols = [c for c in TARGET_VARS if c not in df_rename.columns]
    if missing_cols:
        print(f"    [WARN] Targets {missing_cols} missing in {node_id}. Skipping.")
        continue
        
    # Resample to 1-minute mean
    df_1min = df_rename[TARGET_VARS].resample('1min').mean()
    
    # Track global temporal bounds
    node_min = df_1min.index.min()
    node_max = df_1min.index.max()
    if global_min_time is None or node_min < global_min_time:
        global_min_time = node_min
    if global_max_time is None or node_max > global_max_time:
        global_max_time = node_max
        
    node_data[node_id] = df_1min
    print(f"    Loaded {len(df_1min):,} minutes from {node_min} to {node_max}")

if not node_data:
    raise ValueError("No valid node datasets loaded.")

print(f"Global temporal bounds determined: {global_min_time} to {global_max_time}")

# Reindex all nodes to the global time range
global_time_range = pd.date_range(start=global_min_time, end=global_max_time, freq='1min')
print(f"Global aligned index length: {len(global_time_range):,} rows.")

for nid in list(node_data.keys()):
    node_data[nid] = node_data[nid].reindex(global_time_range)



[1/10] Scanning and loading AWS node datasets...
  Processing id-02 from id-02_raw.csv...
    Loaded 53,774 minutes from 2026-05-14 16:50:00 to 2026-06-21 01:03:00
  Processing id-03 from id-03_raw.csv...
    Loaded 1,327,981 minutes from 2023-12-11 19:13:00 to 2026-06-21 00:13:00
  Processing id-05 from id-05_raw.csv...
    Loaded 660,204 minutes from 2025-03-19 13:41:00 to 2026-06-21 01:04:00
Global temporal bounds determined: 2023-12-11 19:13:00 to 2026-06-21 01:04:00
Global aligned index length: 1,328,032 rows.


# 2. Ingest & Time-Interpolate ERA5 Auxiliary Data


In [5]:
print('\n[2/10] Loading and time-interpolating ERA5 auxiliary data...')
era5_raw = pd.read_csv(PATH_ERA5)
era5_raw['datetime'] = (pd.to_datetime(era5_raw['datetime'], utc=True)
                        .dt.tz_convert('Asia/Jakarta')
                        .dt.tz_localize(None))
era5_raw = (era5_raw.sort_values('datetime')
            .drop_duplicates('datetime')
            .set_index('datetime'))

era5_req_cols = [
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'pressure_msl',
    'wind_speed_10m', 'wind_direction_10m', 'cloud_cover', 'shortwave_radiation', 'rain'
]
# Select columns or fallback if missing
era5_cols = [c for c in era5_req_cols if c in era5_raw.columns]
era5_feats = era5_raw[era5_cols]

# Temporal interpolation to 1-minute global time range
era5_reindexed = era5_feats.reindex(era5_feats.index.union(global_time_range)).sort_index()
era5_interpolated = era5_reindexed.interpolate(method='time')
era5_1min = era5_interpolated.reindex(global_time_range)
era5_1min.columns = [f'era5_{c}' for c in era5_1min.columns]

# Handle edge NaNs
era5_1min = era5_1min.ffill().bfill()
print(f"  ERA5 features processed: {list(era5_1min.columns)}")
print(f"  ERA5 NaNs remaining: {era5_1min.isna().sum().sum()}")



[2/10] Loading and time-interpolating ERA5 auxiliary data...
  ERA5 features processed: ['era5_temperature_2m', 'era5_relative_humidity_2m', 'era5_dew_point_2m', 'era5_surface_pressure', 'era5_wind_speed_10m', 'era5_wind_direction_10m', 'era5_cloud_cover', 'era5_shortwave_radiation', 'era5_rain']
  ERA5 NaNs remaining: 0


# 3. Create Cyclical Temporal Features


In [6]:
print('\n[3/10] Creating cyclical temporal features...')
time_feats = pd.DataFrame(index=global_time_range)
time_feats['hour_sin'] = np.sin(2 * np.pi * global_time_range.hour / 24)
time_feats['hour_cos'] = np.cos(2 * np.pi * global_time_range.hour / 24)
time_feats['doy_sin']  = np.sin(2 * np.pi * global_time_range.dayofyear / 366)
time_feats['doy_cos']  = np.cos(2 * np.pi * global_time_range.dayofyear / 366)
time_feats['month_sin'] = np.sin(2 * np.pi * global_time_range.month / 12)
time_feats['month_cos'] = np.cos(2 * np.pi * global_time_range.month / 12)



[3/10] Creating cyclical temporal features...


# 4. Phase 1: Linear Interpolation (Short Gaps <= 15 min)


In [7]:
print('\n[4/10] Running Phase 1: Linear interpolation on gaps <= 15 minutes...')
node_data_phase1 = {}
for nid, df_node in node_data.items():
    df_p1 = df_node.copy()
    for col in TARGET_VARS:
        before_nans = df_p1[col].isna().sum()
        df_p1[col] = df_p1[col].interpolate(method='linear', limit=15, limit_direction='both')
        after_nans = df_p1[col].isna().sum()
        print(f"  {nid} - {col}: {before_nans:,} NaNs -> {after_nans:,} NaNs (filled {before_nans - after_nans:,})")
    node_data_phase1[nid] = df_p1



[4/10] Running Phase 1: Linear interpolation on gaps <= 15 minutes...
  id-02 - temperature: 1,275,684 NaNs -> 1,275,038 NaNs (filled 646)
  id-02 - humidity: 1,275,684 NaNs -> 1,275,038 NaNs (filled 646)
  id-02 - pressure: 1,275,684 NaNs -> 1,275,038 NaNs (filled 646)
  id-02 - dewpoint: 1,275,684 NaNs -> 1,275,038 NaNs (filled 646)
  id-03 - temperature: 33,656 NaNs -> 13,050 NaNs (filled 20,606)
  id-03 - humidity: 33,656 NaNs -> 13,050 NaNs (filled 20,606)
  id-03 - pressure: 33,656 NaNs -> 13,050 NaNs (filled 20,606)
  id-03 - dewpoint: 33,656 NaNs -> 13,050 NaNs (filled 20,606)
  id-05 - temperature: 679,430 NaNs -> 671,342 NaNs (filled 8,088)
  id-05 - humidity: 679,430 NaNs -> 671,342 NaNs (filled 8,088)
  id-05 - pressure: 679,430 NaNs -> 671,342 NaNs (filled 8,088)
  id-05 - dewpoint: 679,430 NaNs -> 671,342 NaNs (filled 8,088)


# 5. Denoising Autoencoder Setup (Inputs & Masks)


In [8]:
print('\n[5/10] Setting up Denoising Autoencoder inputs and masks...')
node_encoded_dfs = {}
for nid, df_node_p1 in node_data_phase1.items():
    df_node_orig = node_data[nid]
    df_setup = pd.DataFrame(index=global_time_range)
    
    # 1. Observation masks (1 if observed after Phase 1, 0 if missing)
    for col in TARGET_VARS:
        df_setup[f'mask_{col}'] = df_node_p1[col].notna().astype(np.float32)
        
    # 2. Temporary filled target features (fallback to corresponding ERA5 values)
    era5_map = {
        'temperature': 'era5_temperature_2m',
        'humidity': 'era5_relative_humidity_2m',
        'pressure': 'era5_pressure_msl',
        'dewpoint': 'era5_dew_point_2m'
    }
    
    for col in TARGET_VARS:
        # Fill missing values using the corresponding ERA5 proxy
        era_col = era5_map[col] if era5_map[col] in era5_1min.columns else era5_1min.columns[0]
        df_setup[f'filled_{col}'] = df_node_p1[col].fillna(era5_1min[era_col])
        
    # 3. Ground truth targets (keep original observed values before Phase 1 interpolation or after)
    # Note: Target for model optimization should be the original observed value
    for col in TARGET_VARS:
        df_setup[f'target_{col}'] = df_node_p1[col]
        # Weight for training: 1 if observed, 0 if missing
        df_setup[f'weight_{col}'] = df_node_p1[col].notna().astype(np.float32)
        
    # Combine with ERA5 and time features
    df_combined = pd.concat([df_setup, era5_1min, time_feats], axis=1)
    df_combined['node_id'] = nid
    
    # Forward/backward fill all features to prevent NaN features
    feature_cols = [c for c in df_combined.columns if not c.startswith('target_') and c != 'node_id']
    df_combined[feature_cols] = df_combined[feature_cols].ffill().bfill()
    
    node_encoded_dfs[nid] = df_combined

# Fit LabelEncoder for Node ID representation
le_node = LabelEncoder()
le_node.fit(list(node_data.keys()))
joblib.dump(le_node, os.path.join(OUTPUTS_DIR, 'model', 'label_encoder_node.pkl'))
print(f"Node Encoder saved. Classes: {le_node.classes_}")



[5/10] Setting up Denoising Autoencoder inputs and masks...
Node Encoder saved. Classes: ['id-02' 'id-03' 'id-05']


# 6. Extract Strided Sequences


In [ ]:
print('\n[6/10] Extracting training and validation sequences...')
SEQ_LEN = 720
STRIDE_TRAIN = 720
STRIDE_VAL = 720

# Lists for training
X_train_list_cont = []
X_train_list_node = []
Y_train_list = []
W_train_list = []

# Lists for validation
X_val_list_cont = []
X_val_list_node = []
Y_val_list = []
W_val_list = []

# List features
sample_df = next(iter(node_encoded_dfs.values()))
continuous_features = [c for c in sample_df.columns if not c.startswith('target_') and not c.startswith('weight_') and c != 'node_id']
target_cols = [f'target_{col}' for col in TARGET_VARS]
weight_cols = [f'weight_{col}' for col in TARGET_VARS]

print(f"Continuous features count: {len(continuous_features)}")

# Chronological split index
split_point = int(0.8 * len(global_time_range))
train_range = global_time_range[:split_point]
val_range = global_time_range[split_point:]

for nid, df_node in node_encoded_dfs.items():
    node_idx = le_node.transform([nid])[0]
    
    # Split dataframe chronologically
    df_train = df_node.loc[train_range]
    df_val = df_node.loc[val_range]
    
    # 1. Extract training sequences
    cont_arr_train = df_train[continuous_features].values
    target_arr_train = df_train[target_cols].values
    weight_arr_train = df_train[weight_cols].values
    
    for i in range(0, len(df_train) - SEQ_LEN + 1, STRIDE_TRAIN):
        seq_w = weight_arr_train[i:i+SEQ_LEN]
        if seq_w.sum() == 0:
            continue
        X_train_list_cont.append(cont_arr_train[i:i+SEQ_LEN])
        X_train_list_node.append([node_idx])
        Y_train_list.append(target_arr_train[i:i+SEQ_LEN])
        W_train_list.append(seq_w)
        
    # 2. Extract validation sequences (with larger stride for memory efficiency)
    cont_arr_val = df_val[continuous_features].values
    target_arr_val = df_val[target_cols].values
    weight_arr_val = df_val[weight_cols].values
    
    for i in range(0, len(df_val) - SEQ_LEN + 1, STRIDE_VAL):
        seq_w = weight_arr_val[i:i+SEQ_LEN]
        if seq_w.sum() == 0:
            continue
        X_val_list_cont.append(cont_arr_val[i:i+SEQ_LEN])
        X_val_list_node.append([node_idx])
        Y_val_list.append(target_arr_val[i:i+SEQ_LEN])
        W_val_list.append(seq_w)

X_train_cont = np.array(X_train_list_cont, dtype=np.float32)
X_train_node = np.array(X_train_list_node, dtype=np.int32)
Y_train = np.array(Y_train_list, dtype=np.float32)
W_train = np.array(W_train_list, dtype=np.float32)

X_val_cont = np.array(X_val_list_cont, dtype=np.float32)
X_val_node = np.array(X_val_list_node, dtype=np.int32)
Y_val = np.array(Y_val_list, dtype=np.float32)
W_val = np.array(W_val_list, dtype=np.float32)

print(f"Extracted Train samples: {X_train_cont.shape[0]:,}")
print(f"Extracted Val samples: {X_val_cont.shape[0]:,}")

# Free list memory immediately
import gc
del X_train_list_cont, X_train_list_node, Y_train_list, W_train_list
del X_val_list_cont, X_val_list_node, Y_val_list, W_val_list
gc.collect()



[6/10] Extracting training and validation sequences...
Continuous features count: 23
Extracted Train samples: 24,199
Extracted Val samples: 4,846


0

# 7. Scalers & Normalization


In [10]:
print('\n[7/10] Normalizing features...')

# Normalization: Scalers fit on training data
scaler_X = MinMaxScaler()
num_features = X_train_cont.shape[2]
scaler_X.fit(X_train_cont.reshape(-1, num_features))

scaler_Y = MinMaxScaler()
# Fill NaNs temporarily to fit
Y_train_clean = np.nan_to_num(Y_train, nan=0.0)
Y_val_clean = np.nan_to_num(Y_val, nan=0.0)
scaler_Y.fit(Y_train_clean.reshape(-1, len(TARGET_VARS)))

# Save scalers
joblib.dump(scaler_X, os.path.join(OUTPUTS_DIR, 'model', 'scaler_X.pkl'))
joblib.dump(scaler_Y, os.path.join(OUTPUTS_DIR, 'model', 'scaler_Y.pkl'))

# Transform variables in-place to save memory
X_train_cont_s = scaler_X.transform(X_train_cont.reshape(-1, num_features)).reshape(X_train_cont.shape)
X_val_cont_s = scaler_X.transform(X_val_cont.reshape(-1, num_features)).reshape(X_val_cont.shape)

Y_train_s = scaler_Y.transform(Y_train_clean.reshape(-1, len(TARGET_VARS))).reshape(Y_train.shape)
Y_val_s = scaler_Y.transform(Y_val_clean.reshape(-1, len(TARGET_VARS))).reshape(Y_val.shape)

# Free unscaled arrays
del X_train_cont, X_val_cont
gc.collect()

print(f"Train samples scaled: {X_train_cont_s.shape[0]:,}")
print(f"Val samples scaled: {X_val_cont_s.shape[0]:,}")

# Pre-mask training set to simulate missingness dynamically
# For each training sample, mask a random rate of observations in the input features
for s in range(X_train_cont_s.shape[0]):
    # Random missingness rate
    rate = np.random.choice([0.1, 0.2, 0.3, 0.4, 0.5])
    # Identify timesteps where masks are 1 for this sequence (features columns 0-3 are masks)
    for col_idx in range(len(TARGET_VARS)):
        mask_feat_idx = col_idx  # masks are first 4 columns in features
        filled_feat_idx = len(TARGET_VARS) + col_idx # filled targets are next 4 columns
        
        observed_timesteps = np.where(X_train_cont_s[s, :, mask_feat_idx] == 1.0)[0]
        if len(observed_timesteps) > 0:
            num_to_mask = int(rate * len(observed_timesteps))
            mask_times = np.random.choice(observed_timesteps, size=num_to_mask, replace=False)
            
            # Mask features
            X_train_cont_s[s, mask_times, mask_feat_idx] = 0.0  # Set mask to 0
            
            # Fill with ERA5 corresponding feature scaled
            # ERA5 feature column index in features:
            # masks (4) + filled targets (4) + ERA5 (9) + time (6)
            # ERA5 variables start at index 8. Mapping:
            # temperature -> era5_temperature_2m (index 8)
            # humidity -> era5_relative_humidity_2m (index 9)
            # pressure -> era5_pressure_msl (index 11)
            # dewpoint -> era5_dew_point_2m (index 10)
            era_idx_map = {'temperature': 8, 'humidity': 9, 'dewpoint': 10, 'pressure': 11}
            era_col = era_idx_map[TARGET_VARS[col_idx]]
            X_train_cont_s[s, mask_times, filled_feat_idx] = X_train_cont_s[s, mask_times, era_col]



[7/10] Normalizing features...
Train samples scaled: 24,199
Val samples scaled: 4,846


# 8. Build & Train Bidirectional LSTM Model


In [11]:
print('\n[8/10] Building and training Bidirectional LSTM model...')

feat_in = keras.Input(shape=(SEQ_LEN, num_features), name='continuous_input')
node_in = keras.Input(shape=(1,), name='node_input')

# Embedding for Node IDs
num_nodes = len(le_node.classes_)
node_emb = layers.Embedding(input_dim=num_nodes, output_dim=4, name='node_embedding')(node_in)
node_emb = layers.Reshape((4,))(node_emb)
node_emb_rep = layers.RepeatVector(SEQ_LEN)(node_emb)

# Concatenate continuous features and repeated node embeddings
merged = layers.Concatenate(axis=-1)([feat_in, node_emb_rep])

# Bidirectional LSTM Layer Stack
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(merged)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

# Dense Regression Head (predicting 4 targets scaled 0-1)
out = layers.TimeDistributed(layers.Dense(len(TARGET_VARS), activation='sigmoid'), name='imputation_output')(x)

model = keras.Model(inputs=[feat_in, node_in], outputs=out, name='Universal_Imputer_BiLSTM')
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
model.summary()

cb_list = [
    tf_callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor='val_loss'),
    tf_callbacks.ReduceLROnPlateau(patience=6, factor=0.5, monitor='val_loss'),
    tf_callbacks.ModelCheckpoint(os.path.join(OUTPUTS_DIR, 'model', 'best_model_universal.keras'),
                                 save_best_only=True, monitor='val_loss')
]

# Train
history = model.fit(
    x=[X_train_cont_s, X_train_node],
    y=Y_train_s,
    sample_weight=W_train[:, :, 0],  # only calculate loss on observed points
    validation_data=([X_val_cont_s, X_val_node], Y_val_s, W_val[:, :, 0]),
    epochs=20,
    batch_size=256,
    callbacks=cb_list,
    verbose=1
)

# Plot Learning Curve
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss (Weighted MSE)')
plt.plot(history.history['val_loss'], label='Val Loss (Weighted MSE)')
plt.title('Universal Model Learning Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'plots', 'learning_curve.png'), dpi=120)
plt.close()
print("  Learning curve plot saved.")



[8/10] Building and training Bidirectional LSTM model...


I0000 00:00:1781978967.419385      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "Universal_Imputer_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ node_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ node_embedding      │ (None, 1, 4)      │         12 │ node_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 4)         │          0 │ node_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ continuous_input    │ (None, 120, 23)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 120, 4)    │          0 │ reshape[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 120, 27)   │          0 │ continuous_input… │
│ (Concatenate)       │                   │            │ repeat_vector[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 120, 128)  │     47,104 │ concatenate[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 120, 128)  │        512 │ bidirectional[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 120, 128)  │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 120, 64)   │     41,216 │ dropout[0][0]     │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 120, 64)   │        256 │ bidirectional_1[… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 120, 64)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ imputation_output   │ (None, 120, 4)    │        260 │ dropout_1[0][0]   │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 89,360 (349.06 KB)

 Trainable params: 88,976 (347.56 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/20


I0000 00:00:1781978977.537071      66 cuda_dnn.cc:529] Loaded cuDNN version 91002


95/95 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - loss: 0.1043 - mae: 0.2614 - val_loss: 0.0366 - val_mae: 0.1658 - learning_rate: 0.0010
Epoch 2/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0308 - mae: 0.1243 - val_loss: 0.0061 - val_mae: 0.0647 - learning_rate: 0.0010
Epoch 3/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0087 - mae: 0.0671 - val_loss: 0.0026 - val_mae: 0.0455 - learning_rate: 0.0010
Epoch 4/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0050 - mae: 0.0497 - val_loss: 0.0018 - val_mae: 0.0369 - learning_rate: 0.0010
Epoch 5/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0029 - mae: 0.0381 - val_loss: 0.0014 - val_mae: 0.0313 - learning_rate: 0.0010
Epoch 6/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0021 - mae: 0.0327 - val_loss: 8.2086e-04 - val_mae: 0.0250 - learning_rate: 0.0010
Epoch 7/20
95/95 ━━━━━━━━━━━━━━━━━━━━ 6s 61ms/step - loss: 0.0016 - mae: 0.0291 - val_loss: 4.8696e-04 - val_mae: 0.0208 - learning_rate: 0.0010
Epoch 8/20
95/9

# 9. Multi-Rate Simulated Missingness Evaluation


In [12]:
print('\n[9/10] Evaluating model on simulated missingness (10%-50%)...')

rates = [0.1, 0.2, 0.3, 0.4, 0.5]
val_metrics = []

# Define helper metric functions
def calculate_nse(y_true, y_pred):
    num = np.sum((y_true - y_pred) ** 2)
    den = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1.0 - (num / den) if den != 0 else np.nan

def calculate_kge(y_true, y_pred):
    std_true = np.std(y_true)
    std_pred = np.std(y_pred)
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    if std_true == 0 or mean_true == 0:
        return np.nan
    r = np.corrcoef(y_true, y_pred)[0, 1] if std_pred > 0 else 0
    beta = mean_pred / mean_true
    gamma = std_pred / std_true
    return 1.0 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

# Prepare lists to generate report content
report_lines = []
report_lines.append("=== UNIVERSAL IMPUTATION MODEL EVALUATION REPORT ===")
report_lines.append(f"Training nodes: {list(node_data.keys())}")
report_lines.append(f"Lookback window: {SEQ_LEN} minutes\n")

for rate in rates:
    print(f"  Evaluating missingness rate: {int(rate*100)}%...")
    # Copy validation features
    X_val_masked_s = np.copy(X_val_cont_s)
    
    # Record coordinates of masked observed values for metric calculation
    masked_coords = []  # list of tuples: (sample_idx, timestep, target_col)
    
    for s in range(X_val_masked_s.shape[0]):
        for col_idx in range(len(TARGET_VARS)):
            mask_feat_idx = col_idx
            filled_feat_idx = len(TARGET_VARS) + col_idx
            
            # Observe mask values (weight is 1 for observed)
            observed_timesteps = np.where(W_val[s, :, col_idx] == 1.0)[0]
            if len(observed_timesteps) > 0:
                num_to_mask = int(rate * len(observed_timesteps))
                mask_times = np.random.choice(observed_timesteps, size=num_to_mask, replace=False)
                
                # Apply mask to input features
                X_val_masked_s[s, mask_times, mask_feat_idx] = 0.0
                
                era_idx_map = {'temperature': 8, 'humidity': 9, 'dewpoint': 10, 'pressure': 11}
                era_col = era_idx_map[TARGET_VARS[col_idx]]
                X_val_masked_s[s, mask_times, filled_feat_idx] = X_val_masked_s[s, mask_times, era_col]
                
                for t in mask_times:
                    masked_coords.append((s, t, col_idx))
                    
    # Predict
    pred_s = model.predict([X_val_masked_s, X_val_node], batch_size=1024, verbose=0)
    
    # Inverse scale predictions and ground truths
    pred = scaler_Y.inverse_transform(pred_s.reshape(-1, len(TARGET_VARS))).reshape(pred_s.shape)
    y_true_orig = Y_val_clean  # unscaled validation labels
    
    # Extract only the artificially masked points
    # Separate vectors for each variable
    masked_values = {v: {'true': [], 'pred': []} for v in TARGET_VARS}
    
    for (s, t, col_idx) in masked_coords:
        var_name = TARGET_VARS[col_idx]
        masked_values[var_name]['true'].append(y_true_orig[s, t, col_idx])
        masked_values[var_name]['pred'].append(pred[s, t, col_idx])
        
    report_lines.append(f"--- MISSINGNESS RATE: {int(rate*100)}% ---")
    
    for col_idx, var_name in enumerate(TARGET_VARS):
        t_vals = np.array(masked_values[var_name]['true'])
        p_vals = np.array(masked_values[var_name]['pred'])
        
        if len(t_vals) == 0:
            continue
            
        mae = mean_absolute_error(t_vals, p_vals)
        rmse = np.sqrt(mean_squared_error(t_vals, p_vals))
        r2 = r2_score(t_vals, p_vals)
        nse = calculate_nse(t_vals, p_vals)
        kge = calculate_kge(t_vals, p_vals)
        bias = np.mean(p_vals - t_vals)
        pearson_r = np.corrcoef(t_vals, p_vals)[0, 1] if np.std(p_vals) > 0 else 0
        
        val_metrics.append({
            'rate': rate,
            'variable': var_name,
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'nse': nse,
            'kge': kge,
            'bias': bias,
            'pearson_r': pearson_r
        })
        
        report_lines.append(f"Variable: {var_name:<12} | MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f} | NSE: {nse:.4f} | KGE: {kge:.4f} | Bias: {bias:.4f} | R: {pearson_r:.4f}")
        
        # Visualize Observed vs Imputed (Scatter Plot) for 30% rate
        if rate == 0.3:
            # 1. Scatter plot
            plt.figure(figsize=(6, 5))
            plt.scatter(t_vals, p_vals, s=1, alpha=0.3, color='indigo')
            mn, mx = min(t_vals.min(), p_vals.min()), max(t_vals.max(), p_vals.max())
            plt.plot([mn, mx], [mn, mx], 'r--', lw=1.5)
            plt.title(f"Observed vs Imputed: {var_name} (30% Gaps)\nPearson R = {pearson_r:.4f}")
            plt.xlabel("Observed Value")
            plt.ylabel("Imputed Value")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUTS_DIR, 'plots', f'scatter_30pct_{var_name}.png'), dpi=120)
            plt.close()
            
            # 2. Residual Distribution (Density & Histogram)
            residuals = p_vals - t_vals
            plt.figure(figsize=(6, 4))
            plt.hist(residuals, bins=50, density=True, color='teal', alpha=0.7)
            plt.axvline(0, color='red', linestyle='--', lw=1.5)
            plt.title(f"Residual Distribution: {var_name} (30% Gaps)")
            plt.xlabel("Residual (Imputed - Observed)")
            plt.ylabel("Density")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUTS_DIR, 'plots', f'residuals_30pct_{var_name}.png'), dpi=120)
            plt.close()

            # 3. Error Histogram
            plt.figure(figsize=(6, 4))
            plt.hist(np.abs(residuals), bins=50, color='darkorange', alpha=0.7)
            plt.title(f"Absolute Error Histogram: {var_name} (30% Gaps)")
            plt.xlabel("Absolute Error")
            plt.ylabel("Frequency")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUTS_DIR, 'plots', f'error_hist_30pct_{var_name}.png'), dpi=120)
            plt.close()

    # Time series comparison on a gap window for visual inspection (30% missing rate)
    if rate == 0.3:
        # Sample sequence
        sample_seq_idx = 0
        plt.figure(figsize=(15, 10))
        for col_idx, var_name in enumerate(TARGET_VARS):
            plt.subplot(4, 1, col_idx+1)
            t_seq = y_true_orig[sample_seq_idx, :, col_idx]
            p_seq = pred[sample_seq_idx, :, col_idx]
            w_seq = W_val[sample_seq_idx, :, col_idx]
            
            # Plot original observed
            obs_idx = np.where(w_seq == 1.0)[0]
            plt.scatter(obs_idx, t_seq[obs_idx], color='steelblue', s=10, label='Observed')
            
            # Highlight imputed values in masked slots
            # The indices we masked out in this sample
            # Find masked values
            mask_feat_idx = col_idx
            masked_indices = np.where((W_val[sample_seq_idx, :, col_idx] == 1.0) & (X_val_masked_s[sample_seq_idx, :, mask_feat_idx] == 0.0))[0]
            if len(masked_indices) > 0:
                plt.scatter(masked_indices, p_seq[masked_indices], color='darkorange', s=20, marker='x', label='Imputed')
                
            plt.plot(p_seq, color='green', alpha=0.5, linestyle=':', label='Imputation Line')
            plt.ylabel(var_name)
            if col_idx == 0:
                plt.legend(loc='upper right')
        plt.suptitle("Time Series Imputation Sample Comparison (30% Missingness)", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUTS_DIR, 'plots', 'time_series_comparison.png'), dpi=120)
        plt.close()

# Save metrics DataFrame
df_metrics = pd.DataFrame(val_metrics)
df_metrics.to_csv(os.path.join(OUTPUTS_DIR, 'metrics', 'imputation_metrics_summary.csv'), index=False)
print("  Imputation metrics summary CSV saved.")

# Save evaluation text report
with open(os.path.join(OUTPUTS_DIR, 'reports', 'evaluation_report.txt'), 'w') as f_rep:
    f_rep.write("\n".join(report_lines))
print("  Evaluation report text saved.")



[9/10] Evaluating model on simulated missingness (10%-50%)...
  Evaluating missingness rate: 10%...
  Evaluating missingness rate: 20%...
  Evaluating missingness rate: 30%...
  Evaluating missingness rate: 40%...
  Evaluating missingness rate: 50%...
  Imputation metrics summary CSV saved.
  Evaluation report text saved.


# 10. Gap Filling (Inference) and Exporting Final CSVs


In [13]:
print('\n[10/10] Reconstructing actual gaps in all node CSVs...')

for nid, df_node_raw in node_data.items():
    print(f"  Reconstructing targets for {nid}...")
    df_node_p1 = node_data_phase1[nid]
    df_combined = node_encoded_dfs[nid]
    
    # Make sequences for this entire node
    # Slide with stride=1 to cover all points
    n_rows = len(df_combined)
    cont_arr = df_combined[continuous_features].values
    node_idx = le_node.transform([nid])[0]
    
    # Store reconstructed values
    reconstructed_targets = np.copy(df_node_p1[TARGET_VARS].values)
    
    # A single timestep can be covered by multiple overlapping sequence predictions.
    # We aggregate predictions by taking the average of all overlapping windows.
    sum_preds = np.zeros_like(reconstructed_targets)
    count_preds = np.zeros((n_rows, 1))
    
    # We will slide window=120 with stride=1 to predict everything
    # To save memory, we process in chunks of 50000 sequences
    chunk_size = 50000
    for start_idx in range(0, n_rows - SEQ_LEN + 1, chunk_size):
        end_idx = min(start_idx + chunk_size, n_rows - SEQ_LEN + 1)
        
        X_node_cont_list = []
        map_indices = []
        for i in range(start_idx, end_idx):
            X_node_cont_list.append(cont_arr[i:i+SEQ_LEN])
            map_indices.append(i)
            
        X_node_cont_arr = np.array(X_node_cont_list, dtype=np.float32)
        X_node_cont_arr_s = scaler_X.transform(X_node_cont_arr.reshape(-1, num_features)).reshape(X_node_cont_arr.shape)
        X_node_id_arr = np.full((X_node_cont_arr_s.shape[0], 1), node_idx, dtype=np.int32)
        
        preds_s = model.predict([X_node_cont_arr_s, X_node_id_arr], batch_size=4096, verbose=0)
        preds = scaler_Y.inverse_transform(preds_s.reshape(-1, len(TARGET_VARS))).reshape(preds_s.shape)
        
        for idx, start_t in enumerate(map_indices):
            sum_preds[start_t:start_t+SEQ_LEN] += preds[idx]
            count_preds[start_t:start_t+SEQ_LEN] += 1.0
            
        # Free memory of chunk arrays explicitly
        del X_node_cont_list, X_node_cont_arr, X_node_cont_arr_s, X_node_id_arr, preds_s, preds
        import gc
        gc.collect()
        
    avg_preds = np.divide(sum_preds, count_preds, out=np.zeros_like(sum_preds), where=count_preds > 0)
    
    # Clip average predictions to realistic physical ranges
    BOUNDS = {
        'temperature': (10.0, 50.0),
        'humidity': (15.0, 100.0),
        'pressure': (960.0, 1040.0),
        'dewpoint': (5.0, 35.0)
    }
    for col_idx, var_name in enumerate(TARGET_VARS):
        avg_preds[:, col_idx] = np.clip(avg_preds[:, col_idx], *BOUNDS[var_name])
        
    # Replace NaNs in resampled dataset with predictions
    for col_idx, var_name in enumerate(TARGET_VARS):
        mask_nan = np.isnan(reconstructed_targets[:, col_idx])
        reconstructed_targets[mask_nan, col_idx] = avg_preds[mask_nan, col_idx]
        
    # Fallback ffill/bfill for any edge points missed by sequence overlap
    df_imputed = pd.DataFrame(reconstructed_targets, index=global_time_range, columns=TARGET_VARS)
    df_imputed = df_imputed.ffill().bfill()
    
    # Verify no NaNs remain
    n_nans_left = df_imputed.isna().sum().sum()
    print(f"    NaNs left in target columns after LSTM: {n_nans_left}")
    
    # Export full timeline CSV matching the original format
    raw_path = os.path.join(CACHE_DIR, f'{nid}_raw.csv')
    df_raw_template = pd.read_csv(raw_path)
    
    df_raw_template['datetime'] = (pd.to_datetime(df_raw_template['timestamp'], unit='s', utc=True)
                                   .dt.tz_convert('Asia/Jakarta').dt.tz_localize(None))
    
    df_raw_template = (df_raw_template.drop_duplicates(subset='datetime')
                       .set_index('datetime')
                       .sort_index()
                       .reindex(global_time_range))
    
    # Fill target variables
    # The original CSV header names might be 'dew' instead of 'dewpoint'
    inv_col_map = {v: k for k, v in col_map.items()}
    for var_name in TARGET_VARS:
        # Check what the column name was in original CSV
        orig_col = None
        for k, v in col_map.items():
            if v == var_name and k in df_raw_template.columns:
                orig_col = k
                break
        if orig_col is None:
            # Fallback to standard name
            orig_col = var_name
            
        df_raw_template[orig_col] = df_imputed[var_name]
        
    # Set data source label
    # original if observed, linear_interpolated if filled by linear interpolation, ann_lstm_imputed otherwise
    # Let's compare with original node df
    df_node_orig = node_data[nid]
    df_node_p1 = node_data_phase1[nid]
    
    df_raw_template['data_source'] = 'lstm_imputed'
    
    # If observed in df_node_orig, it is 'original'
    orig_observed = df_node_orig[TARGET_VARS[0]].notna()
    df_raw_template.loc[orig_observed, 'data_source'] = 'original'
    
    # If observed in df_node_p1 but not in df_node_orig, it is 'linear_interpolated'
    interp_observed = (~orig_observed) & df_node_p1[TARGET_VARS[0]].notna()
    df_raw_template.loc[interp_observed, 'data_source'] = 'linear_interpolated'
    
    # Reset index and rebuild timestamp
    df_out = df_raw_template.reset_index().rename(columns={'index': 'datetime'})
    df_out['timestamp'] = df_out['datetime'].apply(
        lambda x: int(pd.Timestamp(x).tz_localize('Asia/Jakarta').tz_convert('UTC').timestamp())
        if pd.notnull(x) else None
    )
    
    # Remove temporary datetime column
    df_out = df_out.drop(columns=['datetime'])
    
    # Save file
    out_path = os.path.join(WRITE_CACHE_DIR, f'{nid}_imputed.csv')
    df_out.to_csv(out_path, index=False)
    print(f"    Exported imputed data to {out_path} ({len(df_out):,} rows)")

print('\n=== UNIVERSAL IMPUTATION PROCESS COMPLETED SUCCESSFULLY! ===')



[10/10] Reconstructing actual gaps in all node CSVs...
  Reconstructing targets for id-02...
    NaNs left in target columns after LSTM: 0
    Exported imputed data to /kaggle/working/cache_data/id-02_imputed.csv (1,328,032 rows)
  Reconstructing targets for id-03...
    NaNs left in target columns after LSTM: 0
    Exported imputed data to /kaggle/working/cache_data/id-03_imputed.csv (1,328,032 rows)
  Reconstructing targets for id-05...
    NaNs left in target columns after LSTM: 0
    Exported imputed data to /kaggle/working/cache_data/id-05_imputed.csv (1,328,032 rows)

=== UNIVERSAL IMPUTATION PROCESS COMPLETED SUCCESSFULLY! ===
